In [9]:
import boto3
import json
import PyPDF2
import tiktoken

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        text = ""
        for page in pdf_reader.pages:
            text += page.extract_text()
        return text

def count_tokens(text):
    """Count tokens using tiktoken"""
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    return len(tokens)

def get_model_body(model_id, pdf_text):
    """Generate appropriate request body based on model"""
    if "anthropic" in model_id.lower():
        return {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1000,
            "messages": [
                {
                    "role": "user", 
                    "content": f"Please provide a summary of this document:\n\n{pdf_text}"
                }
            ]
        }
    elif "meta.llama" in model_id.lower():
        return {
            "prompt": f"Please provide a summary of this document:\n\n{pdf_text}",
            "max_gen_len": 1000,
            "temperature": 0.7
        }
    else:
        return {
            "prompt": f"Please provide a summary of this document:\n\n{pdf_text}"
        }

def parse_response(model_id, response_body):
    """Parse response based on model type"""
    if "anthropic" in model_id.lower():
        return response_body['content'][0]['text']
    elif "meta.llama" in model_id.lower():
        return response_body['generation']
    else:
        return str(response_body)

def test_bedrock_with_pdf(pdf_filename, model_id, region, token_limit):
    """Extract PDF text, check tokens, and invoke Bedrock model if under limit"""
    
    # Extract text from PDF
    print("Extracting text from PDF...")
    pdf_text = extract_text_from_pdf(pdf_filename)
    print(f"Extracted {len(pdf_text):,} characters")
    
    # Count tokens
    print("Counting tokens...")
    token_count = count_tokens(pdf_text)
    print(f"Estimated token count: {token_count:,}")
    
    # Apply 10% buffer for output tokens
    effective_limit = int(token_limit * 0.9)
    print(f"Effective limit (90% of {token_limit:,}): {effective_limit:,}")
    
    # Check token limit
    if token_count > effective_limit:
        print(f"Token limit exceeded! {token_count:,} > {effective_limit:,}")
        print("Cannot proceed to model.")
        return None
    
    print(f"Within token limit ({token_count:,}/{effective_limit:,}). Proceeding to model...")
    
    # Initialize Bedrock client
    bedrock = boto3.client('bedrock-runtime', region_name=region)
    
    # Prepare request body
    body = get_model_body(model_id, pdf_text)
    
    try:
        # Invoke model
        print(f"Invoking {model_id} in {region}...")
        response = bedrock.invoke_model(
            modelId=model_id,
            body=json.dumps(body)
        )
        
        # Parse response
        response_body = json.loads(response['body'].read())
        model_response = parse_response(model_id, response_body)
        
        print("Model Response:")
        print("-" * 50)
        print(model_response)
        print("-" * 50)
        
        return model_response
        
    except Exception as e:
        print(f"Error invoking model: {str(e)}")
        return None

# Install required packages
try:
    import PyPDF2
except ImportError:
    print("Installing PyPDF2...")
    !pip install PyPDF2
    import PyPDF2

try:
    import tiktoken
except ImportError:
    print("Installing tiktoken...")
    !pip install tiktoken
    import tiktoken

# Run tests
print("TESTING PDF WITH CLAUDE")
print("=" * 60)
test_bedrock_with_pdf(
    pdf_filename="test_200k_words.pdf", 
    model_id="anthropic.claude-3-5-sonnet-20240620-v1:0",
    region="us-east-1",
    token_limit=200000
)

print("\nTESTING PDF WITH LLAMA")
print("=" * 60)
test_bedrock_with_pdf(
    pdf_filename="test_200k_words.pdf", 
    model_id="us.meta.llama4-maverick-17b-instruct-v1:0",
    region="us-east-1",
    token_limit=1000000
)

TESTING PDF WITH CLAUDE
Extracting text from PDF...
Extracted 1,045,529 characters
Counting tokens...
Estimated token count: 176,291
Effective limit (90% of 200,000): 180,000
Within token limit (176,291/180,000). Proceeding to model...
Invoking anthropic.claude-3-5-sonnet-20240620-v1:0 in us-east-1...
Model Response:
--------------------------------------------------
This document appears to be a 200,000 word test document designed to evaluate token limits and performance of large language models. The key points are:

1. It is generated programmatically to reach approximately 200,000 words for comprehensive testing.

2. It repeats several key phrases throughout, including:
   - Testing with realistic document sizes helps identify potential issues before production deployment
   - Large language models have context windows that define how much text they can process at once
   - Token limits are important considerations when building conversational AI applications
   - Understanding mode

"The document is a test file designed to evaluate token limits in large language models, specifically\nAnthropic's Claude, which is accessible through AWS Bedrock. The purpose of this document is to reach\napproximately 200,000 words for comprehensive testing. The document is generated programmatically\nto ensure consistent testing conditions.\n\nThe main topics discussed in the document are:\n1. Token limits in large language models: The document highlights the importance of understanding\ntoken limits when building conversational AI applications.\n2. Context windows in large language models: The document explains that large language models have\ncontext windows that define how much text they can process at once.\n3. Performance testing: The document emphasizes the need for performance testing to include edge\ncases like maximum context window usage.\n4. Proper error handling: The document stresses the importance of proper error handling when\nworking with token-limited models.\n5. AW